# Retrieval + LoRA Analysis (v2)

Evaluates cluster retrieval quality and command-target complexity for LoRA training.

In [ ]:
import json
import random
from pathlib import Path
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from inference import ClusterBasedToolSystem


In [ ]:
DATA_PATH = ROOT / 'data/man/nl_command_pairs_flat_train_v2.jsonl'
PHASE1 = ROOT / 'checkpoints/intent_embedder/best_model.pt'
PHASE2 = ROOT / 'checkpoints/cluster_retrieval/best_model.pt'
SEED = 42
N_EVAL = 1000
DEVICE = 'cuda:0' if torch.cuda.is_available() else None

print('data:', DATA_PATH)
print('phase1:', PHASE1)
print('phase2:', PHASE2)

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows = load_jsonl(DATA_PATH)
random.seed(SEED)
random.shuffle(rows)
rows = rows[:N_EVAL]
print('eval rows:', len(rows))

In [ ]:
system = ClusterBasedToolSystem.from_pretrained(
    intent_embedder_path=str(PHASE1),
    query_encoder_path=str(PHASE2),
    device=DEVICE,
)
print('system loaded')

In [ ]:
correct = 0
conf = []
errors = []

for r in rows:
    out = system.predict(r['nl_query'], top_k=1, similarity_threshold=0.0, force_tool_call=False)
    pred_tool = out.tool_name
    gold_tool = r['tool']
    conf.append(float(out.confidence))
    ok = pred_tool == gold_tool
    correct += int(ok)
    if not ok:
        errors.append({
            'query': r['nl_query'],
            'gold_tool': gold_tool,
            'pred_tool': pred_tool,
            'confidence': float(out.confidence),
            'cluster_id': int(out.cluster_id),
        })

acc = correct / max(1, len(rows))
print('Top-1 tool accuracy:', round(acc, 4))
print('Mean confidence:', round(float(np.mean(conf)), 4))
print('Errors:', len(errors))

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(conf, bins=30)
plt.title('Retrieval confidence distribution')
plt.xlabel('confidence')
plt.ylabel('count')
plt.tight_layout()
plt.show()

errors_sorted = sorted(errors, key=lambda x: x['confidence'], reverse=True)
errors_sorted[:20]

In [ ]:
# LoRA target complexity diagnostics (full command vs tail).
def command_tail(tool, cmd):
    parts = cmd.strip().split(maxsplit=1)
    if len(parts) == 1:
        return '<NO_ARGS>' if parts[0] == tool else cmd.strip()
    return parts[1].strip() if parts[0] == tool else cmd.strip()

full_lens = [len(r['command'].split()) for r in rows]
tail_lens = [len(command_tail(r['tool'], r['command']).split()) for r in rows]

print('avg full command tokens:', round(float(np.mean(full_lens)), 3))
print('avg tail tokens:', round(float(np.mean(tail_lens)), 3))

plt.figure(figsize=(8,4))
plt.hist(full_lens, bins=30, alpha=0.6, label='full')
plt.hist(tail_lens, bins=30, alpha=0.6, label='tail')
plt.legend()
plt.title('LoRA target length distribution')
plt.xlabel('token count (whitespace split)')
plt.ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
# Save retrieval errors for further analysis
OUT = ROOT / 'output/retrieval_errors_v2.json'
OUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(errors, f, indent=2, ensure_ascii=False)
print('saved:', OUT)